<a href="https://colab.research.google.com/github/Jyotisrj/sql-sales-data-analysis-/blob/main/01_python_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-commerce Delivery & Logistics Performance Analysis

**Role:** Python Analysis

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

**Dataset:** Brazilian E-Commerce (Olist)

## Objectives

* Analyse order and delivery performance.
* Calculate average delivery time and on-time/late delivery rates.
* Identify monthly order trends and order status distribution.
* Analyse freight costs and state-wise delivery performance.
* Explore customer review scores in relation to delivery performance.
* Export summary tables and charts for the Power BI dashboard.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import os
import zipfile

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Python analysis environment is ready!")

Python analysis environment is ready!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_DIR = Path(
    "/content/drive/MyDrive/Ecommerce_Project/olist_cleaned_data"
)

print("Folder exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    print("Files in folder:")
    for file in DATA_DIR.glob("*.csv"):
        print(file.name)

Folder exists: True
Files in folder:
olist_customers_dataset.csv
olist_order_items_dataset.csv
olist_orders_dataset.csv
olist_sellers_dataset.csv


In [4]:
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

print("Datasets loaded successfully!")

print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Customers:", customers.shape)
print("Sellers:", sellers.shape)

Datasets loaded successfully!
Orders: (99441, 8)
Order items: (112650, 7)
Customers: (99441, 5)
Sellers: (3095, 4)


In [5]:
datasets = {
    "Orders": orders,
    "Order Items": order_items,
    "Customers": customers,
    "Sellers": sellers
}

for name, data in datasets.items():
    print(f"\n--- {name} ---")
    print("Rows and columns:", data.shape)
    print("Duplicate rows:", data.duplicated().sum())
    print("Missing values:\n", data.isnull().sum())
    print("Columns:", data.columns.tolist())


--- Orders ---
Rows and columns: (99441, 8)
Duplicate rows: 0
Missing values:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

--- Order Items ---
Rows and columns: (112650, 7)
Duplicate rows: 0
Missing values:
 order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

--- Customers

In [6]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders["order_status"] = (
    orders["order_status"].astype("string").str.strip().str.lower()
)

print("Date columns converted successfully.")

Date columns converted successfully.


In [7]:

# Cell 8 — Calculate average delivery time

# Step 1: Select orders that have been delivered
delivered_orders = orders[orders["order_status"] == "delivered"].copy()

# Step 2: Convert date columns into datetime format
delivered_orders["order_purchase_timestamp"] = pd.to_datetime(
    delivered_orders["order_purchase_timestamp"],
    errors="coerce"
)

delivered_orders["order_delivered_customer_date"] = pd.to_datetime(
    delivered_orders["order_delivered_customer_date"],
    errors="coerce"
)

# Step 3: Remove rows where either date is missing or invalid
delivered_orders = delivered_orders.dropna(
    subset=[
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]
)

# Step 4: Calculate delivery time in days
delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

# Step 5: Calculate average delivery time
average_delivery_days = delivered_orders["delivery_days"].mean()

print("Delivered orders with valid dates:", len(delivered_orders))
print("Average delivery time:", round(average_delivery_days, 2), "days")

Delivered orders with valid dates: 96470
Average delivery time: 12.56 days
